# Deep Hedging

A hedging policy is a neural network mapping market state to position size, trained by stochastic gradient descent on a convex risk measure of terminal profit and loss over simulated paths. Classical delta hedging assumes frictionless complete markets and derives the hedge analytically. Deep hedging drops both assumptions. Transaction costs, discrete rebalancing, and unhedgeable state enter the simulator, and the optimiser finds the policy the market actually rewards.

This notebook trains and evaluates the framework end to end. Sections cover path simulation with exact replay, the risk objective, training under proportional costs, comparison against the Black-Scholes delta baseline, the no-trade band the learned policy opens around the model hedge, stochastic volatility with variance-aware features and a tradable variance swap, barrier liabilities, and deep BSDE pricing. Every quantitative claim in the library is pinned by a test against a closed form or a statistical relationship. The notebook shows the same checks interactively, and the experiments directory holds the full studies whose records regenerate every table and figure.

In [ ]:
import matplotlib.pyplot as plt
import torch

from deephedging import (
    BSDEConfig,
    BSDEProblem,
    CorrelatedGBMSimulator,
    CVaR,
    DeepBSDESolver,
    EuropeanCall,
    FeedForwardPolicy,
    GBMSimulator,
    GeometricBasketCall,
    HestonSimulator,
    HestonVarianceSwapSimulator,
    MertonSimulator,
    MultiAssetFeatures,
    NoiseSpec,
    ProportionalCost,
    RunningMaxFeatures,
    SingleAssetPayoff,
    TrainConfig,
    UpAndOutCall,
    VarianceFeatures,
    bs_call_delta,
    bs_call_price,
    delta_hedge_positions,
    hedge_pnl,
    merton_call_price,
    pnl_from_positions,
    pnl_summary,
    train,
    train_bsde,
)
from deephedging.calibration import (
    CalibrationConfig,
    HestonParams,
    calibrate_heston,
    price_surface,
)
from deephedging.market import kernels_available
from deephedging.pricing import MonteCarloPricer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FUSED = kernels_available()
print(f"device {DEVICE}, fused kernels {FUSED}")

## Market simulation

Paths are generated on the fly each training batch, so data is unbounded and nothing overfits a stored dataset. The GBM sampler is exact in distribution. It evolves the log price, which keeps floating point error growth at the square root of the step count and avoids the biased increment dropping of a multiplicative recursion in reduced precision.

A `NoiseSpec` pair `(seed, stream)` names every draw. The same pair reproduces the same paths bit for bit on a given backend, each training iteration owns its own child stream, and the planned CUDA generator consumes the pair as its Philox subsequence. Replay of any single batch is therefore exact.

In [ ]:
SIGMA, MATURITY, STRIKE, N_STEPS = 0.2, 0.25, 100.0, 30
sim = GBMSimulator(s0=100.0, sigma=SIGMA, maturity=MATURITY, n_steps=N_STEPS, device=DEVICE)

state = sim.simulate(50, noise=NoiseSpec(seed=7))
replay = sim.simulate(50, noise=NoiseSpec(seed=7))
assert torch.equal(state.spot, replay.spot)

grid = torch.linspace(0.0, MATURITY, N_STEPS + 1)
plt.figure(figsize=(7, 3))
plt.plot(grid, state.spot.cpu(), linewidth=0.7, alpha=0.7)
plt.xlabel("time (years)")
plt.ylabel("spot")
plt.title("GBM paths, exact replay verified")
plt.tight_layout()

### Fused generation

The eager simulators pay one kernel launch per tensor operation per step. The fused CUDA generators carry the whole recursion in registers, one thread per path, one launch per batch, with the noise pair mapped onto Philox subsequences so replay stays exact. Measured on this machine they reach several billion GBM paths per second and roughly two hundred times the eager Heston rate. Parity with the eager oracles is distributional and pinned by the test suite.

In [ ]:
if FUSED:
    import time

    from deephedging.market import CudaGBMSimulator, CudaHestonSimulator

    fused_gbm = CudaGBMSimulator(s0=100.0, sigma=SIGMA, maturity=MATURITY, n_steps=N_STEPS)
    fused_heston = CudaHestonSimulator(
        s0=100.0,
        v0=0.04,
        kappa=1.5,
        theta=0.04,
        xi=0.5,
        rho=-0.7,
        maturity=MATURITY,
        n_steps=N_STEPS,
    )
    n_bench = 262_144
    for name, generator_sim in (("gbm", fused_gbm), ("heston", fused_heston)):
        generator_sim.simulate(n_bench, noise=NoiseSpec(seed=1))
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(10):
            generator_sim.simulate(n_bench, noise=NoiseSpec(seed=1))
        torch.cuda.synchronize()
        rate = 10 * n_bench / (time.perf_counter() - start) / 1e6
        print(f"fused {name}: {rate:,.0f}M paths/s")
else:
    print("no CUDA toolchain; eager simulators remain the only backend here")

## Liability, frictions, objective

The desk sells a European call, collects the Black-Scholes premium, and hedges in the underlying. Each trade costs a fixed fraction of traded notional. Terminal PnL per path is premium plus trading gains minus costs minus the payoff.

The training objective is conditional value at risk of the loss via the Rockafellar-Uryasev form. Minimising `w + E[(L - w)+] / (1 - alpha)` jointly over the threshold `w` and the policy equals minimising CVaR. The identity is pointwise in the policy, so a nonconvex network does not break it. The threshold is a learned parameter warm-started at the empirical quantile, never a per-batch quantile, because per-batch inner minimisation is optimistically biased by Jensen and a parameter stays synchronised under distributed training. Only the worst `(1 - alpha)` fraction of paths carries gradient, so batch sizes scale with `1 / (1 - alpha)`.

In [ ]:
payoff = EuropeanCall(strike=STRIKE)
cost = ProportionalCost(rate=2e-3)
PREMIUM = float(bs_call_price(100.0, STRIKE, SIGMA, MATURITY))
print(f"premium {PREMIUM:.4f}")

## Training

The episode engine runs a sequential time-major loop. At each rebalancing date the feature map builds the observation, the policy emits a position, gains and costs accrue. The default observation is log moneyness, time to maturity, and current position, which is the Markovian sufficient state for vanilla liabilities under GBM.

On a CUDA device the whole iteration, forward and backward through the loss, is captured once in a CUDA graph and replayed per batch. The loop is dispatch bound at this network width, with the host issuing tens of microsecond kernels while the device idles, and replay collapses the iteration into one launch. First-iteration losses match the eager path to one part in a hundred thousand; over a trajectory the capture pins kernel algorithm choices that eager re-selects, so equivalence is numerical rather than bitwise.

In [ ]:
torch.manual_seed(11)
policy = FeedForwardPolicy(hidden_sizes=(64, 64)).to(DEVICE)
config = TrainConfig(
    n_iterations=600,
    batch_paths=16384,
    lr=1e-3,
    seed=4,
    graph_episode=(DEVICE == "cuda"),
)
result = train(sim, policy, payoff, cost, CVaR(alpha=0.95), config, premium=PREMIUM)

plt.figure(figsize=(7, 3))
plt.plot(result.losses, linewidth=0.8)
plt.xlabel("iteration")
plt.ylabel("CVaR objective")
plt.title("training loss")
plt.tight_layout()

## Evaluation against baselines

Three books on identical out-of-sample paths. No hedge keeps the premium and eats the payoff. The Black-Scholes delta hedge is optimal in the frictionless continuous limit but pays costs for every rebalance and ignores them when choosing positions. The trained policy knows the costs exist.

Expected shortfall here is the evaluation estimator, clamped at the sample maximum because quantile interpolation plus tail rescaling can otherwise exceed the worst observed loss at extreme levels. Keep it out of training losses.

In [ ]:
eval_state = sim.simulate(100_000, noise=NoiseSpec(seed=99))
with torch.no_grad():
    deep_pnl = hedge_pnl(eval_state, policy, payoff, cost, premium=PREMIUM)
deltas = delta_hedge_positions(eval_state.spot, STRIKE, SIGMA, MATURITY)
delta_pnl = pnl_from_positions(eval_state, deltas, payoff, cost, premium=PREMIUM)
naked_pnl = PREMIUM - payoff(eval_state.spot)

for name, pnl in (("no hedge", naked_pnl), ("BS delta", delta_pnl), ("deep", deep_pnl)):
    print(f"{name:9s} {pnl_summary(pnl)}")

plt.figure(figsize=(7, 3))
for name, pnl in (("BS delta", delta_pnl), ("deep", deep_pnl)):
    plt.hist(pnl.cpu().numpy(), bins=200, range=(-4, 3), alpha=0.5, label=name, density=True)
plt.xlabel("terminal PnL")
plt.legend()
plt.title("hedged PnL under proportional costs")
plt.tight_layout()

The delta hedge trades aggressively because nothing charges it for turnover, so costs eat its premium and widen its left tail. The trained policy under-hedges relative to delta and earns a tighter loss tail at the chosen confidence level. In the frictionless limit the two coincide, which the test suite checks by a dispersion bound on the discrete delta hedge.

## The no-trade band

Under proportional costs the optimal policy holds a band of inventories around the model delta and trades only at the edges. Whalley and Wilmott derived the asymptotic band by singular perturbation, with half-width growing as the cube root of the cost rate. The exponent comes from balancing turnover cost against tracking risk, so it survives the change from exponential utility to the CVaR objective trained here. Sweeping the held position at fixed spot and time exposes the band directly. Where the response curve hugs the diagonal the policy holds, and where it bends away the policy trades back toward the edge. The frontier study stores this probe for every trained run, and `experiments/band_scaling.py` fits the width against cost across the grid.

In [ ]:
delta_atm = float(bs_call_delta(100.0, STRIKE, SIGMA, 0.5 * MATURITY))
inventory = torch.linspace(delta_atm - 0.6, delta_atm + 0.6, 101, device=DEVICE)
probe_features = torch.stack(
    (torch.zeros_like(inventory), torch.full_like(inventory, 0.5), inventory), dim=-1
)
with torch.no_grad():
    response, _ = policy(probe_features, None)

plt.figure(figsize=(5, 4))
plt.plot(inventory.cpu(), response.cpu(), label="policy target")
plt.plot(inventory.cpu(), inventory.cpu(), linestyle="--", linewidth=0.8, label="hold")
plt.axhline(delta_atm, linestyle=":", linewidth=0.8, label="BS delta")
plt.xlabel("held position")
plt.ylabel("new position")
plt.title("no-trade band at the money, mid horizon")
plt.legend()
plt.tight_layout()

## Stochastic volatility

Heston dynamics make variance a second state variable. The simulator uses the full-truncation Euler scheme, the standard bias-minimising discretisation when the Feller condition fails, and exposes the clamped variance path as a named channel on the market state. `VarianceFeatures` appends it to the observation, so the policy can distinguish calm from stressed regimes at the same spot level. A spot-only policy cannot represent that distinction.

In [ ]:
heston = HestonSimulator(
    s0=100.0,
    v0=0.04,
    kappa=1.5,
    theta=0.04,
    xi=0.5,
    rho=-0.7,
    maturity=MATURITY,
    n_steps=N_STEPS,
    device=DEVICE,
)
torch.manual_seed(12)
vol_policy = FeedForwardPolicy(n_features=4, hidden_sizes=(64, 64)).to(DEVICE)
vol_config = TrainConfig(n_iterations=600, batch_paths=16384, lr=1e-3, seed=5)
vol_result = train(
    heston,
    vol_policy,
    payoff,
    cost,
    CVaR(alpha=0.95),
    vol_config,
    premium=PREMIUM,
    feature_map=VarianceFeatures(),
)

heston_eval = heston.simulate(100_000, noise=NoiseSpec(seed=98))
with torch.no_grad():
    vol_pnl = hedge_pnl(
        heston_eval, vol_policy, payoff, cost, premium=PREMIUM, feature_map=VarianceFeatures()
    )
print(f"heston variance-aware {pnl_summary(vol_pnl)}")

## Trading the variance swap

Observing the variance narrows the policy's uncertainty but cannot remove the risk, because the spot alone does not span volatility moves. A variance swap does. Its time-t value is the accrued realised variance plus the CIR conditional expectation of the remainder, affine in the instantaneous variance, so it prices off the same simulated grid that drives the spot. `HestonVarianceSwapSimulator` exposes it as a second asset on the trailing axis, `SingleAssetPayoff` keeps the call liability on the spot column, and the vector policy chooses both positions jointly. The comparison below is paired, with the same evaluation seed driving both books, so the gap is the value of the instrument rather than sampling noise. The full study with seeds and cost levels lives in `experiments/multi_instrument.py`.

In [ ]:
two_asset = HestonVarianceSwapSimulator(heston=heston, vs_maturity=2.0 * MATURITY)
swap_call = SingleAssetPayoff(inner=payoff)
print(f"swap value at inception {two_asset.initial_swap_value():.4f}")

torch.manual_seed(16)
span_features = MultiAssetFeatures(n_assets=2)
span_policy = FeedForwardPolicy(
    n_features=span_features.n_features, hidden_sizes=(64, 64), n_outputs=2
).to(DEVICE)
span_config = TrainConfig(n_iterations=600, batch_paths=16384, lr=1e-3, seed=6)
train(
    two_asset,
    span_policy,
    swap_call,
    cost,
    CVaR(alpha=0.95),
    span_config,
    premium=PREMIUM,
    feature_map=span_features,
)

two_eval = two_asset.simulate(100_000, noise=NoiseSpec(seed=98))
with torch.no_grad():
    span_pnl = hedge_pnl(
        two_eval, span_policy, swap_call, cost, premium=PREMIUM, feature_map=span_features
    )
print(f"spot only      {pnl_summary(vol_pnl)}")
print(f"spot plus swap {pnl_summary(span_pnl)}")

## Jumps

Merton dynamics add a compensated compound-Poisson jump component. Each step samples the exact transition law: the conditional jump sum given the count is a single Gaussian draw, and counts invert the truncated Poisson distribution function against the seeded uniform stream because the library Poisson sampler accepts no generator and would break exact replay. The cumulative jump count ships as a state channel. Conditioning on the count makes the terminal price lognormal, so the call price is a Poisson-weighted Black series, the golden reference the Monte Carlo estimate must hit.

In [ ]:
merton = MertonSimulator(
    s0=100.0,
    sigma=0.15,
    jump_intensity=1.0,
    jump_mean=-0.1,
    jump_vol=0.15,
    maturity=1.0,
    n_steps=50,
    device=DEVICE,
)
merton_estimate = MonteCarloPricer(n_paths=200_000, seed=131).price(
    EuropeanCall(strike=100.0), merton
)
merton_reference = float(
    merton_call_price(
        spot=100.0,
        strike=100.0,
        sigma=0.15,
        jump_intensity=1.0,
        jump_mean=-0.1,
        jump_vol=0.15,
        tau=1.0,
    )
)
print(
    f"merton call  series {merton_reference:.4f}  "
    f"monte carlo {merton_estimate.value:.4f} +- {merton_estimate.standard_error:.4f}"
)
jump_state = merton.simulate(50_000, noise=NoiseSpec(seed=137))
print(f"mean jumps per path {float(jump_state.aux['jumps'][-1].mean()):.3f} (intensity 1.0)")

## Multi-asset books

The spot grid carries a trailing asset axis, positions become vectors, and gains and costs contract that axis per step. The correlated simulator mixes independent drivers through the Cholesky factor of a validated correlation matrix, leaving each marginal exactly lognormal. The geometric basket call stays lognormal under correlation, so its closed form is the multi-asset golden, and the cross-sectional feature map shows the policy every asset's log moneyness and held position at once.

In [ ]:
basket_sim = CorrelatedGBMSimulator(
    s0=100.0,
    sigmas=(0.2, 0.3),
    correlation=((1.0, 0.5), (0.5, 1.0)),
    maturity=MATURITY,
    n_steps=10,
    device=DEVICE,
)
basket_payoff = GeometricBasketCall(strike=100.0)
basket_premium = MonteCarloPricer(n_paths=200_000, seed=139).price(basket_payoff, basket_sim)
print(f"basket premium {basket_premium.value:.4f} +- {basket_premium.standard_error:.4f}")

torch.manual_seed(14)
basket_features = MultiAssetFeatures(n_assets=2)
basket_policy = FeedForwardPolicy(
    n_features=basket_features.n_features, hidden_sizes=(32, 32), n_outputs=2
).to(DEVICE)
basket_config = TrainConfig(n_iterations=300, batch_paths=8192, lr=2e-3, seed=15)
train(
    basket_sim,
    basket_policy,
    basket_payoff,
    cost,
    CVaR(alpha=0.95),
    basket_config,
    premium=basket_premium.value,
    feature_map=basket_features,
)
basket_eval = basket_sim.simulate(100_000, noise=NoiseSpec(seed=149))
with torch.no_grad():
    basket_pnl = hedge_pnl(
        basket_eval,
        basket_policy,
        basket_payoff,
        cost,
        premium=basket_premium.value,
        feature_map=basket_features,
    )
naked_basket = basket_premium.value - basket_payoff(basket_eval.spot)
print(f"vector hedge {pnl_summary(basket_pnl)}")
print(f"no hedge     {pnl_summary(naked_basket)}")

## Barrier liabilities

An up-and-out call knocks out when the spot touches the barrier at any monitoring date, inception included, so no closed-form hedge exists and the practitioner fallback is the vanilla delta, which neither sees the barrier nor stops trading after knockout. The running maximum is the sufficient path statistic for the knockout, cached on the market state as a cumulative maximum so the episode pays linear rather than quadratic cost in the horizon. `RunningMaxFeatures` feeds it to the policy, which learns to stop hedging a dead contract. Both books below receive the same Monte Carlo premium and score on the same paths. The full study under Heston with both deep arms lives in `experiments/barrier_hedging.py`.

In [ ]:
barrier_payoff = UpAndOutCall(strike=STRIKE, barrier=115.0)
barrier_estimate = MonteCarloPricer(n_paths=200_000, seed=97).price(barrier_payoff, sim)
print(
    f"up-and-out premium {barrier_estimate.value:.4f} "
    f"+- {barrier_estimate.standard_error:.4f}"
)

torch.manual_seed(17)
barrier_policy = FeedForwardPolicy(n_features=4, hidden_sizes=(64, 64)).to(DEVICE)
barrier_config = TrainConfig(n_iterations=600, batch_paths=16384, lr=1e-3, seed=8)
train(
    sim,
    barrier_policy,
    barrier_payoff,
    cost,
    CVaR(alpha=0.95),
    barrier_config,
    premium=barrier_estimate.value,
    feature_map=RunningMaxFeatures(),
)

barrier_eval = sim.simulate(100_000, noise=NoiseSpec(seed=96))
barrier_deltas = delta_hedge_positions(barrier_eval.spot, STRIKE, SIGMA, MATURITY)
barrier_delta_pnl = pnl_from_positions(
    barrier_eval, barrier_deltas, barrier_payoff, cost, premium=barrier_estimate.value
)
with torch.no_grad():
    barrier_pnl = hedge_pnl(
        barrier_eval,
        barrier_policy,
        barrier_payoff,
        cost,
        premium=barrier_estimate.value,
        feature_map=RunningMaxFeatures(),
    )
print(f"vanilla delta {pnl_summary(barrier_delta_pnl)}")
print(f"deep barrier  {pnl_summary(barrier_pnl)}")

## Deep BSDE pricing

High-dimensional semilinear pricing PDEs reformulate as backward SDEs. The solver learns the inception value `Y0` and a network for the volatility process `Z`, evolves `Y` forward by explicit Euler, and minimises the terminal mismatch against the payoff. For a generator uniformly Lipschitz in `(Y, Z)` the achieved loss is an a posteriori bound on the solution error, so the final loss certifies accuracy rather than merely indicating convergence. Scope is the semilinear Lipschitz class. Proportional costs, gamma constraints, and uncertain volatility leave it and need reflected or second-order BSDE machinery.

With the zero generator the solved `Y0` is the plain expectation, here the undiscounted Black-Scholes price, and the learned `Z` at inception approximates `sigma * S0 * delta`. The dimension sweep in `experiments/bsde_pricing.py` prices the geometric basket call to fifty dimensions against its lognormal closed form, where the cost of a grid method would be astronomical and the solver's grows linearly.

In [ ]:
torch.manual_seed(13)
problem = BSDEProblem(
    dim=1,
    x0=100.0,
    sigma=0.2,
    maturity=1.0,
    n_steps=10,
    terminal=lambda x: torch.clamp(x[:, 0] - 100.0, min=0.0),
)
solver = DeepBSDESolver(dim=1, hidden_sizes=(32, 32)).to(DEVICE)
bsde_config = BSDEConfig(n_iterations=800, batch_paths=512, lr=3e-3, seed=7)
bsde_result = train_bsde(problem, solver, bsde_config)
reference = float(bs_call_price(100.0, 100.0, 0.2, 1.0))
print(f"deep BSDE price {bsde_result.y0:.4f}  Black-Scholes {reference:.4f}")
print(f"terminal-matching loss {bsde_result.final_loss:.5f} (a posteriori error certificate)")

## Surface calibration

The Heston characteristic function in the trap-safe Gatheral form prices a whole strike grid through the COS expansion, one batched contraction with nothing adaptive, so prices are differentiable in the model parameters. Calibration minimises vega-weighted price residuals, first-order equal to implied-volatility residuals without an inner root-find, over a multi-maturity surface. A single strip underdetermines the parameters because the smile constrains a blend of initial and long-run variance; the term structure separates them, which the synthetic recovery below demonstrates from a deliberately distant start.

In [ ]:
true_params = HestonParams(v0=0.045, kappa=2.0, theta=0.05, xi=0.4, rho=-0.6)
strikes = torch.tensor([80.0, 90.0, 95.0, 100.0, 105.0, 110.0, 120.0], dtype=torch.float64)
maturities = (0.25, 1.0, 3.0)
surface = torch.stack(
    [price_surface(true_params.as_tensors(), 100.0, strikes, tau) for tau in maturities]
)
start = HestonParams(v0=0.09, kappa=1.0, theta=0.02, xi=0.7, rho=-0.2)
calibration = calibrate_heston(
    surface, 100.0, strikes, maturities, start, CalibrationConfig(n_iterations=600, lr=5e-2)
)
print(f"loss {calibration.losses[0]:.3e} -> {calibration.final_loss:.3e}")
print(f"true      {true_params}")
print(f"recovered {calibration.params}")

## Design notes

Decisions fixed by adversarial verification rather than convention. Antithetic variates are off by default because a good hedge drives residual PnL toward an even function of the noise, where pairing doubles variance exactly when training has converged. Control variates live in evaluation only, since a policy-independent additive control provably does nothing for gradients. The CVaR threshold is a parameter warm-started at the empirical quantile, never a batch quantile. Whole-episode graph capture is the sound capture unit; capturing the per-step policy corrupts gradients through clobbered static activations, which measurement confirmed before the design settled. Mixed precision keeps state in fp32 log space with bf16 confined to network matmuls, and pays only for networks wide enough to engage tensor cores. The variance swap accrues realised variance by the trapezoid rule and values the remaining leg from the same clamped grid, so its price increments carry no convention mismatch a policy could learn to arbitrage.

The experiments directory holds the full studies behind these sections. The frontier sweeps cost and risk aversion against the delta hedge and stores the no-trade band probe per run, the band-scaling fit tests the Whalley-Wilmott cube-root law across the cost grid, the barrier study isolates the value of the running-maximum observation, the instrument-span study measures what the variance swap removes from the tail, and the BSDE sweep prices to fifty dimensions. Every run appends a JSON line with commit, library versions, device, seeds, and losses, so each table regenerates from its store.

Next. A backward pass that regenerates paths from Philox offsets instead of storing the grid, per-asset cost rates for realistic swap frictions, and multi-GPU scaling over disjoint noise streams.